# Stage 7: Filter Duplicates

**Purpose**: Identifies and marks duplicate items that share identical `resumen` or `texto` content.

**Input**: `data/intermediate/OBRA CIVIL/OBRA CIVIL_stage5.json`  
**Output**: `data/intermediate/OBRA CIVIL/OBRA CIVIL_stage6.json` → `_stage7.json`

## What this notebook does

Parametric expansion can occasionally produce items with identical text content (e.g., when different parameter combinations yield the same final description). This stage:

1. **Indexes items** by their `resumen` and `texto` content
2. **Detects duplicates** where multiple item keys share identical text
3. **Marks items** with a `validation` flag (`true` = unique, `false` = duplicate)
4. **Filters output** to retain only unique items

### Why this matters

For retrieval evaluation, each query (resumen) should map to exactly one target (texto). Duplicates would create ambiguous ground truth and inflate metrics artificially.

### Output

Items marked `validation: false` are excluded from the final dataset.

In [29]:
import json
from pathlib import Path
from collections import defaultdict

def marcar_duplicados(path_entrada: str, path_salida: str) -> None:
    with open(path_entrada, encoding="utf-8") as f:
        data = json.load(f)

    resumen_map = defaultdict(list)
    texto_map = defaultdict(list)

    # Index by resumen and texto
    for key, item in data.items():
        resumen = item.get("resumen", "").strip()
        texto = item.get("texto", "").strip()
        resumen_map[resumen].append(key)
        texto_map[texto].append(key)

    # Detect duplicates
    duplicados = set()
    for mapa in (resumen_map, texto_map):
        for lista in mapa.values():
            if len(lista) > 1:
                duplicados.update(lista)

    # Add validation property
    for key in data:
        data[key]["validation"] = key not in duplicados

    # Save updated JSON
    with open(path_salida, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

    print(f"Archivo actualizado con 'validation'. Guardado en {path_salida}")

In [30]:
from utils import config

marcar_duplicados(
    path_entrada=config.stage_path("OBRA CIVIL", 5),   # ← cambia el chapter aquí
    path_salida=config.stage_path("OBRA CIVIL", 6),
)

Archivo actualizado con 'validation'. Guardado en /work/data/intermediate/OBRA CIVIL/OBRA CIVIL_stage6.json


In [31]:
# This script takes data from the _stage6 pipeline and prints out a sample according to its 'validation' flag
import sys
from pathlib import Path
import json
import random
from collections import Counter, defaultdict
import pandas as pd
from utils import config


def main(chapter):
    document_path = config.stage_path(chapter, 6)

    # Load JSON file
    with open(document_path, "r", encoding="utf-8") as file:
        raw_data = json.load(file)

    data = {k: v for k, v in raw_data.items() if not v.get("validation", True)}

    # Group by 'resumen' to identify unique values
    resumen_counts = defaultdict(list)

    for key, item in data.items():
        resumen_counts[item["resumen"]].append(key)

    # Keep only unique resumen values
    unique_resumen_items = {keys[0]: data[keys[0]] for resumen, keys in resumen_counts.items() if len(keys) == 1}

    # Select a random sample of unique resumen items (adjust sample size as needed)
    sample_size = min(3, len(unique_resumen_items))  # Ensure we don't exceed available unique items
    sample_items = random.sample(list(unique_resumen_items.items()), sample_size)

    # Convert to a display-friendly format
    sample_result = [{key: item} for key, item in sample_items]

    # Convert to a DataFrame and display
    df_sample = pd.DataFrame([{**{"ID": key}, **item} for key, item in sample_items])
    df_sample = df_sample[['ID', 'concept', 'resumen', 'texto', 'validation']]  # Selecting relevant columns

    # Display in Jupyter Notebook
    return df_sample
    

In [32]:
df=main("OBRA CIVIL")

In [33]:
df

,ID,concept,resumen,texto,validation
0,ODA090abc,SUPLEMENTO POR HUMEDAD EXCESIVA,Suplemento por humedad cuando en obra se super...,Suplemento por humedad cuando en obra se super...,False
1,OEA050hdaa,CANALETA DE HORMIGÓN,Canaleta de hormigón tipo ADIF (1000x450x290)...,Canaleta prefabricada de hormigón armado de 25...,False
2,OAD110aabb,GRAVA,Grava de 20-40 mm de dimensiones. (D/3 a 5/E),Grava de diámetro 20-40 mm para formación de r...,False


# Duplicates Analysis

The Parse Duplicates Code below takes the _stage6 pipeline data, finds duplicate items and dumps them into a _duplicates file. 

## Duplicates Analysis Code

In [34]:
# This code finds duplicated items in the _stage6 pipeline and dumps them into a _duplicate_ file

import sys
from pathlib import Path
import json
from collections import Counter, defaultdict
from utils import config


def main(chapter):
    document_path = config.stage_path(chapter, 6)

    # Load JSON file
    with open(document_path, "r", encoding="utf-8") as file:
        raw_data = json.load(file)

    data = {k: v for k, v in raw_data.items()}
    
    # Extract 'resumen' and 'texto' with their keys
    resumen_dict = {}
    texto_dict = {}
    validation_dict = {}

    for key, value in data.items():
        resumen_dict[key] = value.get("resumen", "").strip()
        texto_dict[key] = value.get("texto", "").strip()
        validation_dict[key] = value.get("validation")

    # Count total items
    total_resumen = len(resumen_dict)
    total_texto = len(texto_dict)

    # Count validated true items
    total_validated_true_resumen = sum(1 for k in resumen_dict if validation_dict[k])
    total_validated_true_texto = sum(1 for k in texto_dict if validation_dict[k])
    
    # Count unique items
    unique_resumen = len(set(resumen_dict.values()))
    unique_texto = len(set(texto_dict.values()))

    # Count unique validated true items
    unique_validated_true_resumen = len(set(resumen for k, resumen in resumen_dict.items() if validation_dict[k]))
    unique_validated_true_texto = len(set(texto for k, texto in texto_dict.items() if validation_dict[k]))

    # Find duplicate items
    resumen_counter = Counter(resumen_dict.values())
    texto_counter = Counter(texto_dict.values())

    # Group duplicates
    duplicate_resumen = defaultdict(list)
    duplicate_texto = defaultdict(list)

    duplicate_validated_true_resumen = 0
    duplicate_validated_false_resumen = 0
    duplicate_validated_true_texto = 0
    duplicate_validated_false_texto = 0

    # Track keys that are duplicates in either field
    either_duplicate_keys = set()

    for key, resumen in resumen_dict.items():
        if resumen_counter[resumen] > 1:  # Only keep duplicates
            duplicate_resumen[resumen].append({"key": key, "validation": validation_dict[key]})
            either_duplicate_keys.add(key)
            if validation_dict[key]:
                duplicate_validated_true_resumen += 1
            else:
                duplicate_validated_false_resumen += 1

    for key, texto in texto_dict.items():
        if texto_counter[texto] > 1:  # Only keep duplicates
            duplicate_texto[texto].append({"key": key, "validation": validation_dict[key]})
            either_duplicate_keys.add(key)
            if validation_dict[key]:
                duplicate_validated_true_texto += 1
            else:
                duplicate_validated_false_texto += 1

    total_duplicates_resumen = duplicate_validated_true_resumen + duplicate_validated_false_resumen
    total_duplicates_texto = duplicate_validated_true_texto + duplicate_validated_false_texto

    # Create a dict of items that are duplicates in either field
    either_duplicate_items = {}
    either_duplicate_validated_true = 0
    either_duplicate_validated_false = 0

    for key in either_duplicate_keys:
        either_duplicate_items[key] = {
            "resumen": resumen_dict[key],
            "texto": texto_dict[key],
            "validation": validation_dict[key],
            "is_resumen_duplicate": resumen_counter[resumen_dict[key]] > 1,
            "is_texto_duplicate": texto_counter[texto_dict[key]] > 1
        }
        
        if validation_dict[key]:
            either_duplicate_validated_true += 1
        else:
            either_duplicate_validated_false += 1
    
    # Convert duplicates to JSON format
    resumen_duplicates_json = {resumen: keys for resumen, keys in duplicate_resumen.items()}
    texto_duplicates_json = {texto: keys for texto, keys in duplicate_texto.items()}

    # Save to JSON files
    duplicate_resumen_path = config.INTERMEDIATE_DIR / chapter / f"{chapter}_duplicate_resumen.json"
    with open(duplicate_resumen_path, "w", encoding="utf-8") as file:
        json.dump(resumen_duplicates_json, file, indent=4, ensure_ascii=False)

    duplicate_texto_path = config.INTERMEDIATE_DIR / chapter / f"{chapter}_duplicate_texto.json"
    with open(duplicate_texto_path, "w", encoding="utf-8") as file:
        json.dump(texto_duplicates_json, file, indent=4, ensure_ascii=False)

    # Save either_duplicate_items to JSON file
    either_duplicate_path = config.INTERMEDIATE_DIR / chapter / f"{chapter}_either_duplicate.json"
    with open(either_duplicate_path, "w", encoding="utf-8") as file:
        json.dump(either_duplicate_items, file, indent=4, ensure_ascii=False)

    # Print results
    print(f"Total 'resumen' items: {total_resumen}")
    print(f"Total 'duplicated' items': {total_duplicates_resumen}")
    print(f"Duplicates - validated true 'resumen' items: {duplicate_validated_true_resumen}")
    print(f"Duplicates - validated false 'resumen' items: {duplicate_validated_false_resumen}")
    print(f"Unique 'resumen' items: {unique_resumen}")
    print(f"Total validated true 'resumen' items: {total_validated_true_resumen}")
    print(f"Unique validated true 'resumen' items: {unique_validated_true_resumen}, {unique_validated_true_resumen/total_validated_true_resumen:.2f}")
    print(f"Duplicate 'resumen' groups: {len(duplicate_resumen)} (saved in 'duplicate_resumen.json')")

    print(f"\nTotal 'texto' items: {total_texto}")
    print(f"Total 'duplicated' items': {total_duplicates_texto}")
    print(f"Duplicates - validated true 'texto' items: {duplicate_validated_true_texto}")
    print(f"Duplicates - validated false 'texto' items: {duplicate_validated_false_texto}")
    print(f"Unique 'texto' items: {unique_texto}")
    print(f"Total validated true 'texto' items: {total_validated_true_texto}")
    print(f"Unique validated true 'texto' items: {unique_validated_true_texto}, {unique_validated_true_texto/total_validated_true_texto:.2f}")
    print(f"Duplicate 'texto' groups: {len(duplicate_texto)} (saved in 'duplicate_texto.json')")

    print(f"\nItems that are duplicates in EITHER 'resumen' OR 'texto': {len(either_duplicate_keys)}")
    print(f"Either duplicate - validated true items: {either_duplicate_validated_true}")
    print(f"Either duplicate - validated false items: {either_duplicate_validated_false}")
    print(f"Either duplicate items saved in '{chapter}_either_duplicate.json'")

In [35]:
main("OBRA CIVIL")

Total 'resumen' items: 126938
Total 'duplicated' items': 13866
Duplicates - validated true 'resumen' items: 0
Duplicates - validated false 'resumen' items: 13866
Unique 'resumen' items: 118570
Total validated true 'resumen' items: 111644
Unique validated true 'resumen' items: 111644, 1.00
Duplicate 'resumen' groups: 5498 (saved in 'duplicate_resumen.json')

Total 'texto' items: 126938
Total 'duplicated' items': 8492
Duplicates - validated true 'texto' items: 0
Duplicates - validated false 'texto' items: 8492
Unique 'texto' items: 121785
Total validated true 'texto' items: 111644
Unique validated true 'texto' items: 111644, 1.00
Duplicate 'texto' groups: 3339 (saved in 'duplicate_texto.json')

Items that are duplicates in EITHER 'resumen' OR 'texto': 15294
Either duplicate - validated true items: 0
Either duplicate - validated false items: 15294
Either duplicate items saved in 'OBRA CIVIL_either_duplicate.json'


## JSON filter script
This code will generate a _stage6 file with unique texto & resumen items

In [36]:
import json
from utils import config

def filter_json(json_data):
    """
    Filter JSON data based on validation flag and uniqueness of resumen and texto fields.
    
    Args:
        json_data (dict): The original JSON data
        
    Returns:
        dict: Filtered JSON with only validated items and no duplicates
    """
    result = {}
    seen_resumens = set()
    seen_textos = set()
    
    # Iterate through all items in the original JSON
    for key, item in json_data.items():
        # Check if validation is true
        if item.get('validation') == True:
            resumen = item.get('resumen')
            texto = item.get('texto')
            
            # Check if we've already seen this resumen or texto
            if resumen not in seen_resumens and texto not in seen_textos:
                # Add to our tracking sets
                seen_resumens.add(resumen)
                seen_textos.add(texto)
                
                # Add to result
                result[key] = item
    
    return result

def main(chapter):
    document_path = config.stage_path(chapter, 6)

    # Load JSON file
    with open(document_path, "r", encoding="utf-8") as file:
        data = json.load(file)

    # Filter the data
    filtered_data = filter_json(data)

    # Save the filtered data to a new file
    output_path = config.stage_path(chapter, 7)
    with open(output_path, "w", encoding="utf-8") as file:
        json.dump(filtered_data, file, indent=4, ensure_ascii=False)
    
    print(f"Unique items: {len(filtered_data)} (saved in {output_path})")

In [37]:
main('OBRA CIVIL')

Unique items: 111644 (saved in /work/data/intermediate/OBRA CIVIL/OBRA CIVIL_stage7.json)
